In [2]:
from pathlib import Path

BASE_PATH = Path(r"C:\Users\Sahil\Women-safety meraki")

print("Project folder exists :", BASE_PATH.exists())
print("src/ exists           :", (BASE_PATH / "src").exists())
print("src/__init__.py exists:", (BASE_PATH / "src" / "__init__.py").exists())
print("src/config.py exists  :", (BASE_PATH / "src" / "config.py").exists())
print("src/h3_utils.py exists:", (BASE_PATH / "src" / "h3_utils.py").exists())

print("\n📂 What is actually inside the project folder:")
for p in sorted(BASE_PATH.iterdir()):
    print("   ", ("📁 " if p.is_dir() else "📄 ") + p.name)

# Is some OTHER 'src' being picked up?
try:
    import src
    print("\n⚠️ A module named 'src' is importable from:", src.__file__)
except ImportError:
    print("\n(no 'src' module importable yet — expected if files don't exist)")

Project folder exists : True
src/ exists           : False
src/__init__.py exists: False
src/config.py exists  : False
src/h3_utils.py exists: False

📂 What is actually inside the project folder:
    📁 .ipynb_checkpoints
    📄 02_ncrb_india_data.ipynb
    📄 03_osm_poi_features.ipynb
    📄 04_weather_data.ipynb
    📄 Advance models.ipynb
    📄 aws s3 upload .ipynb
    📄 Baseline.ipynb
    📄 chicago crime.ipynb
    📁 data
    📁 docs
    📄 ED.ipynb
    📄 Feature engineering.ipynb
    📄 Untitled.ipynb
    📄 Untitled1.ipynb
    📄 Untitled2.ipynb

(no 'src' module importable yet — expected if files don't exist)


In [3]:
from pathlib import Path

BASE_PATH = Path(r"C:\Users\Sahil\Women-safety meraki")
SRC = BASE_PATH / "src"
SRC.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- __init__.py
(SRC / "__init__.py").write_text(
    "# Meraki - Predictive Women's Safety Intelligence Platform\n__version__ = '1.0.0'\n",
    encoding="utf-8"
)

# ------------------------------------------------------------------ config.py
CONFIG_CODE = """# Meraki — Central Configuration
# All hardcoded values live here. Import from anywhere in the codebase.
import os
from pathlib import Path

# ---- PROJECT PATHS (auto-detected from this file's location) ----
PROJECT_ROOT  = Path(__file__).resolve().parent.parent
DATA_DIR      = PROJECT_ROOT / 'data'
RAW_DIR       = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR    = DATA_DIR / 'models'
DOCS_DIR      = PROJECT_ROOT / 'docs'
FIGURES_DIR   = DOCS_DIR / 'figures'

for _d in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, FIGURES_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ---- AWS ----
AWS_REGION            = 'ap-south-1'
BUCKET_NAME           = os.environ.get('MERAKI_BUCKET', 'meraki-2026')
ATHENA_RESULTS_BUCKET = os.environ.get('MERAKI_ATHENA_BUCKET', 'meraki-2026-athena-results')
RISK_CACHE_TABLE      = 'meraki-risk-cache'
ALERTS_TABLE          = 'meraki-alerts'
USERS_TABLE           = 'meraki-users'

# ---- ML ----
H3_RESOLUTION         = 9          # ~250m hexagonal cells
TIME_WINDOW_HOURS     = 3          # 3-hour prediction windows
TRAIN_TEST_SPLIT_DATE = '2022-10-01'

# ---- CITY BOUNDING BOXES ----
BBOX_CHICAGO = {
    'south': 41.6445, 'north': 42.0230,
    'west': -87.9401, 'east': -87.5240,
    'center_lat': 41.8781, 'center_lng': -87.6298,
}
BBOX_DELHI = {
    'south': 28.4040, 'north': 28.8840,
    'west': 76.8380, 'east': 77.3460,
    'center_lat': 28.6139, 'center_lng': 77.2090,
}

# ---- CRIME CATEGORIES ----
WOMEN_SAFETY_TYPES = [
    'ASSAULT', 'BATTERY', 'CRIM SEXUAL ASSAULT',
    'SEX OFFENSE', 'STALKING', 'KIDNAPPING',
    'HOMICIDE', 'ROBBERY', 'INTIMIDATION',
    'HUMAN TRAFFICKING', 'OFFENSE INVOLVING CHILDREN',
]

NCRB_DELHI_2022 = {
    'Rape': 1229,
    'Attempt to commit Rape': 157,
    'Kidnapping & Abduction': 3588,
    'Dowry Deaths': 117,
    'Assault on Women': 2541,
    'Insult to Modesty': 578,
    'Cruelty by Husband/Relatives': 4674,
    'Stalking': 489,
    'Voyeurism': 128,
    'Cyber Crimes against Women': 348,
    'Human Trafficking': 8,
    'Others': 390,
}
"""
(SRC / "config.py").write_text(CONFIG_CODE, encoding="utf-8")

# ---------------------------------------------------------------- h3_utils.py
H3_CODE = """# Meraki — H3 v4 helper functions
import h3
from typing import List, Tuple


def latlng_to_cell(lat: float, lng: float, resolution: int = 9) -> str:
    '''Convert lat/lng to an H3 cell id (v4 API).'''
    return h3.latlng_to_cell(lat, lng, resolution)


def cell_to_latlng(cell: str) -> Tuple[float, float]:
    '''Centre (lat, lng) of an H3 cell (v4 API).'''
    return h3.cell_to_latlng(cell)


def get_neighbors(cell: str, k: int = 1) -> List[str]:
    '''All cells within k rings of the given cell (includes the cell itself).'''
    return list(h3.grid_disk(cell, k))


def cell_area_km2(cell: str) -> float:
    '''Area of an H3 cell in square km.'''
    return h3.cell_area(cell, unit='km^2')


def vectorized_latlng_to_cell(lats, lngs, resolution: int = 9) -> List[str]:
    '''Fast H3 assignment for whole columns (10x faster than df.apply).'''
    return [h3.latlng_to_cell(lat, lng, resolution) for lat, lng in zip(lats, lngs)]
"""
(SRC / "h3_utils.py").write_text(H3_CODE, encoding="utf-8")

print("✅ Created:")
for f in sorted(SRC.iterdir()):
    print("   ", f)

✅ Created:
    C:\Users\Sahil\Women-safety meraki\src\__init__.py
    C:\Users\Sahil\Women-safety meraki\src\config.py
    C:\Users\Sahil\Women-safety meraki\src\h3_utils.py


In [6]:
# --- CELL 1: Setup & Path Fix ---
import sys
import pandas as pd
from pathlib import Path

BASE_PATH = Path(r"C:\Users\Sahil\Women-safety meraki")

if str(BASE_PATH) not in sys.path:
    sys.path.insert(0, str(BASE_PATH))

from src.config import PROCESSED_DIR, FIGURES_DIR, BBOX_CHICAGO, BBOX_DELHI, H3_RESOLUTION
from src.h3_utils import vectorized_latlng_to_cell, get_neighbors

print(f"📁 Project root : {BASE_PATH}")
print(f"📁 Processed dir: {PROCESSED_DIR}")
print(f"📁 Figures dir  : {FIGURES_DIR}")
print("✅ Ready — imports successful!")

📁 Project root : C:\Users\Sahil\Women-safety meraki
📁 Processed dir: C:\Users\Sahil\Women-safety meraki\data\processed
📁 Figures dir  : C:\Users\Sahil\Women-safety meraki\docs\figures
✅ Ready — imports successful!


In [7]:
# --- CELL 2: Download Function with Cache ---

def get_historical_weather(latitude: float, longitude: float,
                            start_date: str, end_date: str,
                            city_name: str, hot_threshold_c: float = 30.0):
    """
    Download hourly historical weather from Open-Meteo.
    Free, no API key, generous rate limits.
    
    ✅ FIX: hot_threshold_c parameterized (30°C Chicago vs 38°C Delhi)
    """
    cache_path = PROCESSED_DIR / f"{city_name.lower()}_weather_hourly.parquet"
    
    # ✅ FIX: Check cache first
    if cache_path.exists():
        print(f"📂 Loading cached {city_name} weather from {cache_path.name}")
        return pd.read_parquet(cache_path)
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m", "relative_humidity_2m",
            "precipitation", "rain", "wind_speed_10m", "is_day"
        ],
        "timezone": "auto"
    }
    
    print(f"⏳ Downloading {city_name} weather ({start_date} to {end_date})...")
    response = requests.get(url, params=params, timeout=180)
    response.raise_for_status()
    data = response.json()
    
    hourly = data['hourly']
    df = pd.DataFrame({
        'datetime': pd.to_datetime(hourly['time']),
        'temperature': hourly['temperature_2m'],
        'humidity': hourly['relative_humidity_2m'],
        'precipitation_mm': hourly['precipitation'],
        'rain_mm': hourly['rain'],
        'wind_speed_kmh': hourly['wind_speed_10m'],
        'is_daylight': hourly['is_day']
    })
    
    # Derived features
    df['hour'] = df['datetime'].dt.hour
    df['date'] = df['datetime'].dt.date
    df['month'] = df['datetime'].dt.month
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['is_raining'] = (df['rain_mm'] > 0.5).astype(int)
    df['is_heavy_rain'] = (df['rain_mm'] > 5.0).astype(int)
    df['is_hot'] = (df['temperature'] > hot_threshold_c).astype(int)
    df['is_dark'] = (1 - df['is_daylight']).astype(int)
    
    # Save to cache
    df.to_parquet(cache_path, index=False)
    print(f"   ✅ {len(df):,} hourly records → cached")
    return df


# --- Download for both cities ---
df_weather_chicago = get_historical_weather(
    latitude=BBOX_CHICAGO['center_lat'],
    longitude=BBOX_CHICAGO['center_lng'],
    start_date="2019-01-01",
    end_date="2023-12-31",
    city_name="chicago",
    hot_threshold_c=30.0  # Chicago summer
)

df_weather_delhi = get_historical_weather(
    latitude=BBOX_DELHI['center_lat'],
    longitude=BBOX_DELHI['center_lng'],
    start_date="2022-01-01",
    end_date="2022-12-31",
    city_name="delhi",
    hot_threshold_c=38.0  # Delhi summer
)

📂 Loading cached chicago weather from chicago_weather_hourly.parquet
📂 Loading cached delhi weather from delhi_weather_hourly.parquet


In [8]:
# --- CELL 3: Quick Summary ---

print("=" * 60)
print("CHICAGO WEATHER")
print("=" * 60)
print(f"Records: {len(df_weather_chicago):,}")
print(f"Rain hours: {df_weather_chicago['is_raining'].sum():,}")
print(f"Hot hours (>30°C): {df_weather_chicago['is_hot'].sum():,}")
print(f"Avg temp: {df_weather_chicago['temperature'].mean():.1f}°C")

print("\n" + "=" * 60)
print("DELHI WEATHER")
print("=" * 60)
print(f"Records: {len(df_weather_delhi):,}")
print(f"Rain hours: {df_weather_delhi['is_raining'].sum():,}")
print(f"Hot hours (>38°C): {df_weather_delhi['is_hot'].sum():,}")
print(f"Avg temp: {df_weather_delhi['temperature'].mean():.1f}°C")

print("\n✅ Notebook 04 complete!")

CHICAGO WEATHER
Records: 43,824
Rain hours: 2,267
Hot hours (>30°C): 0
Avg temp: 10.5°C

DELHI WEATHER
Records: 8,760
Rain hours: 254
Hot hours (>38°C): 482
Avg temp: 24.8°C

✅ Notebook 04 complete!


In [9]:
# Cell 4: Save weather data
base_path = r"C:\Users\Sahil\Women-safety meraki"

# Chicago weather
output_path_chicago = os.path.join(base_path, 'data', 'processed', 'chicago_weather_hourly.parquet')
df_weather_chicago.to_parquet(output_path_chicago, index=False)
print(f"💾 Saved: {output_path_chicago}")

# Delhi weather
output_path_delhi = os.path.join(base_path, 'data', 'processed', 'delhi_weather_hourly.parquet')
df_weather_delhi.to_parquet(output_path_delhi, index=False)
print(f"💾 Saved: {output_path_delhi}")

print("\n💾 Weather data saved successfully!")
print(f"\n📊 Chicago weather: {len(df_weather_chicago):,} hours")
print(f"   Rain hours: {df_weather_chicago['is_raining'].sum():,}")
print(f"   Date range: {df_weather_chicago['datetime'].min()} → "
      f"{df_weather_chicago['datetime'].max()}")

print(f"\n📊 Delhi weather: {len(df_weather_delhi):,} hours") 
print(f"   Rain hours: {df_weather_delhi['is_raining'].sum():,}")
print(f"   Hot hours (>38°C): {df_weather_delhi['is_hot'].sum():,}")

💾 Saved: C:\Users\Sahil\Women-safety meraki\data\processed\chicago_weather_hourly.parquet
💾 Saved: C:\Users\Sahil\Women-safety meraki\data\processed\delhi_weather_hourly.parquet

💾 Weather data saved successfully!

📊 Chicago weather: 43,824 hours
   Rain hours: 2,267
   Date range: 2019-01-01 00:00:00 → 2023-12-31 23:00:00

📊 Delhi weather: 8,760 hours
   Rain hours: 254
   Hot hours (>38°C): 482
